# Report di valutazione

Carica un report di valutazione JSON (prodotto da `italian_llm.evaluation.runner.run_eval`) e ne mostra le metriche in tabella.

Cerca in ordine: `outputs/eval/report.json`, `outputs/eval/eval_report.json`, `data/eval/eval_report.json`. Se nessuno e' presente usa un report dimostrativo, cosi' il notebook resta eseguibile.

In [ ]:
import os
import sys
import json

def find_repo_root(start=None):
    d = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(d, 'src', 'italian_llm')):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return os.path.abspath(os.getcwd())
        d = parent

REPO_ROOT = find_repo_root()
SRC = os.path.join(REPO_ROOT, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

print('Repo root:', REPO_ROOT)

In [ ]:
CANDIDATE_PATHS = [
    os.path.join(REPO_ROOT, 'outputs', 'eval', 'report.json'),
    os.path.join(REPO_ROOT, 'outputs', 'eval', 'eval_report.json'),
    os.path.join(REPO_ROOT, 'data', 'eval', 'eval_report.json'),
]

DEMO_REPORT = {
    'timestamp': '2026-01-01T00:00:00+00:00',
    'eval_set': 'data/eval/eval_it.jsonl (demo)',
    'n_examples': 12,
    'generation_mode': 'mock',
    'model_path': 'Qwen/Qwen2.5-7B',
    'adapter': 'outputs/sft_qwen9b_lora',
    'metrics': {
        'instruction_adherence': 0.71,
        'italianity_score': 0.93,
        'rouge_l': 0.34,
        'verbosity_ratio': 1.12,
        'refusal_rate': 0.08,
        'over_refusal_rate': 0.02,
        'coding_passk': 0.55,
    },
    'latency': {'n_generated': 12, 'mean_s': 0.0, 'p50_s': 0.0, 'p90_s': 0.0, 'total_s': 0.0},
    'by_domain': {
        'qa': {'n': 4, 'italianity': 0.95, 'instruction_adherence': 0.74},
        'coding': {'n': 4, 'italianity': 0.88, 'instruction_adherence': 0.69},
        'email': {'n': 4, 'italianity': 0.96, 'instruction_adherence': 0.70},
    },
}

report = None
report_source = None
for path in CANDIDATE_PATHS:
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as fh:
            report = json.load(fh)
        report_source = path
        break

if report is None:
    report = DEMO_REPORT
    report_source = '(report dimostrativo: nessun file trovato)'

print('Sorgente report:', report_source)
print('Eval set:', report.get('eval_set'))
print('Esempi:', report.get('n_examples'), '| modalita:', report.get('generation_mode'))

In [ ]:
def render_table(headers, table_rows):
    widths = [len(h) for h in headers]
    for row in table_rows:
        for i, cell in enumerate(row):
            widths[i] = max(widths[i], len(str(cell)))
    print('  '.join(h.ljust(widths[i]) for i, h in enumerate(headers)))
    print('  '.join('-' * widths[i] for i in range(len(headers))))
    for row in table_rows:
        print('  '.join(str(cell).ljust(widths[i]) for i, cell in enumerate(row)))

metrics = report.get('metrics', {})
rows_m = []
for name, value in metrics.items():
    if value is None:
        shown = 'n/d'
    elif isinstance(value, float):
        shown = format(value, '.4f')
    else:
        shown = str(value)
    rows_m.append([name, shown])

print('Metriche aggregate:')
print()
render_table(['metrica', 'valore'], rows_m)

In [ ]:
by_domain = report.get('by_domain', {})
if by_domain:
    print('Dettaglio per dominio:')
    print()
    rows_d = []
    for domain, agg in sorted(by_domain.items()):
        ital = agg.get('italianity', 0.0)
        adher = agg.get('instruction_adherence', 0.0)
        rows_d.append([domain, agg.get('n', 0), format(ital, '.3f'), format(adher, '.3f')])
    render_table(['dominio', 'n', 'italianita', 'aderenza'], rows_d)
else:
    print('Nessun dettaglio per dominio nel report.')